In [1]:
import pandas as pd
import googlemaps
import folium
import warnings
import os
from folium.plugins import MarkerCluster

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_excel("../AliancaCentro-Rio/Aliança Centro-Rio 3.0.xlsx")
latlongs = pd.read_excel("../AliancaCentro-Rio/8. LATLONGS/data/ocorrencias.xlsx")

In [3]:
df.drop(columns=["Latitude", "Longitude"], inplace=True)

In [4]:
# remove acentos das colunas "Resposta do 1746" e "Resposta 2
df["Resposta do 1746"] = df["Resposta do 1746"].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.upper()
df["Resposta 2"] = df["Resposta 2"].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.upper()

df["Parecer"] = df["Parecer"].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.upper()
df["Parecer 2"] = df["Parecer 2"].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8').str.upper()

In [5]:
# mesma coisa para "obstaculos"
df_obstaculos_fixos_calcada = df.query("`Tipo de ocorrência` == 'Fiscalização de obstáculo fixo na calçada'")

In [6]:
df_obstaculos_fixos_calcada = pd.merge(df_obstaculos_fixos_calcada, 
                         latlongs[["Número do protocolo", "Latitude", "Longitude"]].drop_duplicates(), 
                         how="left", 
                         on="Número do protocolo")

In [7]:
df_obstaculos_fixos_calcada.drop_duplicates(subset="Número do protocolo", keep="first", inplace=True)

In [8]:
df_obstaculos_fixos_calcada.shape

(116, 33)

Conseguimos fazer um mapa com essas duas ocorrências específicas divididas entre solucionadas, em andamento e fechado ou redirecionada?

In [9]:
df_obstaculos_fixos_calcada["Status"].fillna("-", inplace=True)
df_obstaculos_fixos_calcada["Status 2"].fillna("-", inplace=True)

df_obstaculos_fixos_calcada["Parecer"].fillna("-", inplace=True)
df_obstaculos_fixos_calcada["Parecer 2"].fillna("-", inplace=True)

In [10]:
import numpy as np

In [11]:
df_obstaculos_fixos_calcada["Status_Final"] = [np.nan] * df_obstaculos_fixos_calcada.shape[0]

In [12]:
# for i, row in df_obstaculos_fixos_calcada.iterrows():
#     if row["Parecer 2"] == ""
#     # elif row["Status"] == "Fechado" and row["Parecer"].__contains__("REDIRECIONADO") and row["Status 2"] == "Fechado" and row["Parecer 2"] == "-":
#     #     df_obstaculos_fixos_calcada.at[i, "Tipo"] = "Redirecionado"

In [13]:
for i, row in df_obstaculos_fixos_calcada.iterrows():
    if row["Parecer"] == "SOLUCIONADO":
        df_obstaculos_fixos_calcada.at[i, "Status_Final"] = "Solucionado"
    elif row["Parecer 2"] == "SOLUCIONADO":
        df_obstaculos_fixos_calcada.at[i, "Status_Final"] = "Solucionado"
    elif row["Parecer"].__contains__("REDIRECIONADO") and row["Status 2"] == "Fechado" and row["Parecer 2"] == "-":
        df_obstaculos_fixos_calcada.at[i, "Status_Final"] = "Fechado"
    elif row["Parecer"].__contains__("REDIRECIONADO") and row["Status 2"] == "-" and row["Parecer 2"] == "-":
        df_obstaculos_fixos_calcada.at[i, "Status_Final"] = "Redirecionado"
    elif row["Status 2"] == "Em andamento":
        df_obstaculos_fixos_calcada.at[i, "Status_Final"] = "Em andamento"
    elif row["Status"] == "Fechado" and row["Parecer"] == "-":
        df_obstaculos_fixos_calcada.at[i, "Status_Final"] = "Fechado"
    elif row["Status 2"] == "Fechado" and row["Parecer 2"] == "-":
        df_obstaculos_fixos_calcada.at[i, "Status_Final"] = "Fechado"
    elif row["Status"] == "Em andamento":
        df_obstaculos_fixos_calcada.at[i, "Status_Final"] = "Em andamento"

In [14]:
# checar sempre se tem algum vazio em Status_Final
df_obstaculos_fixos_calcada[["Status", "Parecer", "Status 2", "Parecer 2", "Status_Final"]].drop_duplicates()

,Status,Parecer,Status 2,Parecer 2,Status_Final
0,Fechado,SOLUCIONADO,-,-,Solucionado
4,Fechado,RECORRIDO,Fechado,SOLUCIONADO,Solucionado
27,Fechado,REDIRECIONADO PARA EMPRESA,Fechado,SOLUCIONADO,Solucionado
39,Fechado,RECORRIDO,Em andamento,-,Em andamento
45,Fechado,NOVO CHAMADO,Fechado,SOLUCIONADO,Solucionado
60,Fechado,NOVO CHAMADO,Em andamento,-,Em andamento
72,Em andamento,-,-,SOLUCIONADO,Solucionado
93,Em andamento,-,-,-,Em andamento
110,Em andamento,SOLUCIONADO,-,-,Solucionado


In [15]:
# Create the map
mapa = folium.Map(location=[-22.900252, -43.178084], zoom_start=20, tiles='CartoDB Positron')

# Add title
title_html = '''
             <h3 align="center" style="font-size:20px"><b>Ocorrências de obstáculos fixos na calçada</b></h3>
             '''
mapa.get_root().html.add_child(folium.Element(title_html))

# Create separate marker clusters for each "Status_Final"
marker_cluster_em_andamento = MarkerCluster(name='Em andamento').add_to(mapa)
marker_cluster_solucionado = MarkerCluster(name='Solucionado').add_to(mapa)
marker_cluster_fechado = MarkerCluster(name='Fechado').add_to(mapa)
marker_cluster_redirecionado = MarkerCluster(name='Redirecionado').add_to(mapa)

for i, row in df_obstaculos_fixos_calcada.iterrows():
    protocolo = row['Número do protocolo']
    endereco = row['Endereço']
    ponto_referencia = row['Ponto de referência']
    regiao = row['Região']
    status1 = row['Status']
    parecer = row['Parecer']
    status2 = row['Status 2']
    lat = row['Latitude']
    lng = row['Longitude']
    tipo = row["Status_Final"]

    string = f"""
    Protocolo = {protocolo}<br>
    Endereço = {endereco}<br>
    Ponto de referência = {ponto_referencia}<br>
    Região = {regiao}<br>
    Status 1 = {status1}<br>
    Parecer = {parecer}<br>
    Status 2 = {status2}<br>
    """

    if os.path.isfile(f'fotos/{protocolo}.jpeg'):
        string += f"<img src='fotos/{protocolo}.jpeg'>"
    elif os.path.isfile(f'fotos/{protocolo}.jpg'):
        string += f"<img src='fotos/{protocolo}.jpg'>"
    elif os.path.isfile(f'fotos/{protocolo}.JPEG'):
        string += f"<img src='fotos/{protocolo}.JPEG'>"
    elif os.path.isfile(f'fotos/{protocolo}.JPG'):
        string += f"<img src='fotos/{protocolo}.JPG'>"

    popup = folium.Popup(string, max_width=300, min_width=300)
    if tipo == "Em andamento":
        marker = folium.Marker([lat, lng], popup=popup, icon=folium.Icon(color='orange'))
        marker.add_to(marker_cluster_em_andamento)
    elif tipo == "Solucionado":
        marker = folium.Marker([lat, lng], popup=popup, icon=folium.Icon(color='blue'))
        marker.add_to(marker_cluster_solucionado)
    elif tipo == "Redirecionado":
        marker = folium.Marker([lat, lng], popup=popup, icon=folium.Icon(color='red'))
        marker.add_to(marker_cluster_redirecionado)
    elif tipo == "Fechado":
        marker = folium.Marker([lat, lng], popup=popup, icon=folium.Icon(color='gray'))
        marker.add_to(marker_cluster_fechado)

# Add layer control to the map
folium.LayerControl().add_to(mapa)

mapa.save('index.html')